# 03 - Gene embeddings: all four categories

Item 0 of the brief: a representation for **every** gene in the organism's transcriptome, not only
the measured panel, drawn from four kinds of evidence.

| category | source | what the geometry encodes |
|---|---|---|
| `spatial` | autoencoder on a spatial dataset's `G x N` matrix | co-variation across positioned cells |
| `single_cell` | autoencoder on a dissociated dataset, and the joint models | co-variation across dissociated cells |
| `non_transcriptomic` | ESM-2 over UniProt sequences; STRING PPI spectral embedding | protein sequence; physical/functional interaction |
| `causal` | KEGG signed directed regulatory relations; the perturbation dataset | curated regulatory direction and sign |

**The observation unit, which decides the whole design.** The deliverable is `G x d`, so the
*gene* is the observation and the `N_k` cells are the features. An autoencoder here trains on `G`
samples of dimension `N_k`. Two datasets therefore have different, non-comparable feature spaces -
different cells - and the only pairing that exists across datasets is **the same gene**. That is
why the joint methods pair genes (not cells), and why pairing cells across datasets would be
wrong.

In [ ]:
import os, sys, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))  # import panelprep from the repo root
os.chdir(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else None
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np, pandas as pd
from panelprep import Config
print('panelprep ready; cwd =', os.getcwd())

In [ ]:
from panelprep.embed import EMBEDDERS
EMBEDDERS.meta_table()[['category', 'mode', 'description']]

## Separate vs joint on the synthetic fixture

The fixture generates genes from a known `loadings` matrix, so embedding quality is measurable: does the embedding's k-nearest-neighbour graph recover the true loading geometry?

In [ ]:
from panelprep.io.synthetic import make_synthetic_case
from panelprep.embed import run_embedders, gene_universe

case = make_synthetic_case(seed=0, n_genes=200)
genes = gene_universe(case.candidates, how='intersection')
print(len(genes), 'genes')

embs = run_embedders(case.candidates, embedder_names=['pca_baseline', 'ae_separate', 'vae_separate'],
                     genes=genes, dim=16, seed=0, max_obs=600)
embs.update(run_embedders(case.candidates, embedder_names=['joint_ae', 'dcca', 'contrastive_ae'],
                          genes=genes, dim=16, seed=0, max_obs=600))
pd.DataFrame([{'embedding': k, 'category': v.category, 'shape': str(v.matrix.shape),
               'nan_genes': int(v.matrix.isna().all(axis=1).sum())} for k, v in embs.items()]).set_index('embedding')

In [ ]:
from sklearn.neighbors import NearestNeighbors

def knn_sets(X, k=15):
    X = np.asarray(X, dtype=float)
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(X))).fit(X)
    return [set(r[1:]) for r in nn.kneighbors(X, return_distance=False)]

truth = pd.DataFrame(case.truth['loadings'], index=case.truth['genes']).loc[genes]
truth_knn = knn_sets(truth)

def jaccard_vs_truth(emb, k=15):
    M = emb.matrix.loc[genes].dropna(how='all')
    if M.shape[0] < k + 2:
        return float('nan')
    keep = [genes.index(g) for g in M.index]
    got = knn_sets(M.to_numpy(), k)
    return float(np.mean([len(a & truth_knn[i]) / len(a | truth_knn[i])
                          for a, i in zip(got, keep) if a | truth_knn[i]]))

rng = np.random.default_rng(0)
null = float(np.mean([len(set(rng.choice(len(genes), 15, replace=False)) & t) /
                      len(set(rng.choice(len(genes), 15, replace=False)) | t) for t in truth_knn]))
scores = {k: jaccard_vs_truth(v) for k, v in embs.items()}
scores['__random_baseline__'] = null
pd.Series(scores).sort_values(ascending=False).round(4).to_frame('kNN Jaccard vs true loadings')

## Non-transcriptomic and causal sources (real, mouse)

These ignore the expression matrices entirely and embed the gene list from sequence, interaction and curated-regulation evidence. Coverage is reported, not assumed: genes with no reviewed mouse protein, no STRING node or no KEGG node come back as NaN rows.

In [ ]:
MOUSE_GENES = ['Sox2', 'Pou5f1', 'Nanog', 'T', 'Foxa2', 'Gata4', 'Gata6', 'Hand1', 'Hand2',
               'Mesp1', 'Mixl1', 'Eomes', 'Lefty1', 'Lefty2', 'Nodal', 'Bmp4', 'Wnt3', 'Fgf8',
               'Cer1', 'Sox17', 'Krt8', 'Krt18', 'Mest', 'Dnmt3b', 'Pdgfra', 'Cdx2', 'Tbx6',
               'Msgn1', 'Otx2', 'Hesx1']

ext = run_embedders({}, embedder_names=['esm2_sequence', 'string_ppi', 'kegg_causal'],
                    genes=MOUSE_GENES, dim=16, seed=0)
pd.DataFrame([{'embedding': k, 'category': v.category, 'shape': str(v.matrix.shape),
               'covered': int(v.matrix.notna().any(axis=1).sum()), 'of': v.matrix.shape[0],
               **{kk: vv for kk, vv in v.provenance.items() if isinstance(vv, (int, float, str))}}
              for k, v in ext.items()]).set_index('embedding')

## Pathway-coloured embedding maps

Colour is pathway membership; marker shape is the ligand / receptor / transcription-factor role. The silhouette and kNN-label-agreement numbers next to each panel are what makes this a result rather than a picture.

In [ ]:
from panelprep.annotate import annotate_genes
from panelprep.viz import embedding_panel, embedding_quality

ann = annotate_genes(MOUSE_GENES, sources=['kegg', 'roles'], cache_dir='data/external', offline=False)
display(ann.table.head(10))
all_embs = {**ext}
pd.DataFrame([{'embedding': k, **embedding_quality(v, ann)} for k, v in all_embs.items()]).set_index('embedding')

In [ ]:
os.makedirs('outputs/embeddings/figures', exist_ok=True)
embedding_panel(all_embs, ann, out_path='outputs/embeddings/figures/embedding_panel.png')